Import necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
#import nltk

Load and explore the data

In [2]:
df = pd.read_csv(r"IMDB Dataset.csv")

print("="*50)
print("BASIC INFO")
print("="*50)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 3 reviews:")
display(df.head(3))

print("\n" + "="*50)
print("SENTIMENT DISTRIBUTION")
print("="*50)
print(df['sentiment'].value_counts())
print()
print(df['sentiment'].value_counts(normalize=True) * 100)

print("\n" + "="*50)
print("MISSING VALUES")
print("="*50)
print(df.isnull().sum())

BASIC INFO
Shape: (50000, 2)

Columns: ['review', 'sentiment']

First 3 reviews:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive



SENTIMENT DISTRIBUTION
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

sentiment
positive    50.0
negative    50.0
Name: proportion, dtype: float64

MISSING VALUES
review       0
sentiment    0
dtype: int64


In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK data (run only once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 2. Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 3. Convert to lowercase
    text = text.lower()
    
    # 4. Tokenization
    tokens = word_tokenize(text)
    
    # 5. Remove stopwords + Lemmatization
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens 
        if word not in stop_words and len(word) > 2
    ]
    
    # Join back to string
    return ' '.join(cleaned_tokens)

# Apply the cleaning function (this may take 1-3 minutes)
print("Cleaning reviews... Please wait...")
df['cleaned_review'] = df['review'].apply(preprocess_text)

print("Pre-processing completed!")
print("\nExample of original vs cleaned review:")
print("\nOriginal:")
print(df['review'][0][:300])
print("\nCleaned:")
print(df['cleaned_review'][0][:300])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...


Cleaning reviews... Please wait...
Pre-processing completed!

Example of original vs cleaned review:

Original:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

Cleaned:
one reviewer mentioned watching episode youll hooked right exactly happened methe first thing struck brutality unflinching scene violence set right word trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use wordit called nickname given oswald maximum security s


TF-IDF Vectorization + Train-Test Split

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Convert sentiment to numbers
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)   # keep top 5000 words
X = tfidf.fit_transform(df['cleaned_review'])
y = df['label']

print("TF-IDF shape:", X.shape)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:    ", X_test.shape)

TF-IDF shape: (50000, 5000)
Training set shape: (40000, 5000)
Test set shape:     (10000, 5000)


Train Models

In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Naive Bayes
print("Training Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# 2. SVM
print("Training SVM...")
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)

print("\nBoth models trained successfully!")

Training Naive Bayes...
Training SVM...

Both models trained successfully!


Model Evaluation

In [6]:
def evaluate_model(model, X_test, y_test, model_name):
    print("="*60)
    print(f"EVALUATION: {model_name}")
    print("="*60)
    
    y_pred = model.predict(X_test)
    
    print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()

# Evaluate both models
evaluate_model(nb_model, X_test, y_test, "Naive Bayes")
evaluate_model(svm_model, X_test, y_test, "SVM (LinearSVC)")

EVALUATION: Naive Bayes

Accuracy: 0.8514

Classification Report:
              precision    recall  f1-score   support

    Negative       0.86      0.84      0.85      5000
    Positive       0.85      0.86      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

Confusion Matrix:
[[4221  779]
 [ 707 4293]]

EVALUATION: SVM (LinearSVC)

Accuracy: 0.8776

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.87      0.88      5000
    Positive       0.87      0.88      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000

Confusion Matrix:
[[4362  638]
 [ 586 4414]]



Test the SVM Model with a Custom Review

In [8]:
def predict_sentiment(review):
    # Clean the review using the same pre-processing function
    cleaned = preprocess_text(review)
    
    # Convert to TF-IDF
    vectorized = tfidf.transform([cleaned])
    
    # Predict
    prediction = svm_model.predict(vectorized)[0]
    
    return "Positive" if prediction == 1 else "Negative"

# ---- Test Examples ----
print(predict_sentiment("This movie was absolutely amazing! I loved every minute of it."))
print(predict_sentiment("What a terrible waste of time. The acting was horrible and the story made no sense."))
print(predict_sentiment("It was an okay film, nothing special but not bad either."))

Positive
Negative
Negative
